### Setup

In [ ]:
!apt-get install unrar tree

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unrar x /content/drive/MyDrive/TUKL/data_online_line_width_alpha.rar "*.csv" /content/
!mv /content/data_online_line_width_alpha/ /content/hwd/
!tree -L 1 /content/hwd/

### Data Prep

#### CSV Handling

In [ ]:
import pandas as pd
import glob

main_files = glob.glob('/content/hwd/main_*.csv')
samples = []
for f in main_files:
    df = pd.read_csv(f)
    full_paths = '/content/hwd/' + df['csv']
    samples.extend(full_paths.tolist())

# Write the list to a file that
with open('/content/samples.txt', 'w') as f:
    f.write('\n'.join(samples))

print(f"Extracted {len(samples)} data files")

Extracted 0 data files


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def process_sample(file_path, distance=100) -> np.ndarray | None:
    df = pd.read_csv(file_path, usecols=['X cood.', 'Y cood.', 'Time', 'pen_down'], engine='pyarrow')
    df = df[df['pen_down'] > 0.9]
    df = df.sort_values('Time').reset_index(drop=True)
    df = df.rename(columns={'X cood.': 'x', 'Y cood.': 'y'})

    if len(df) < 2:
        return None

    xy = df[['x', 'y']].to_numpy(dtype=np.float32)
    kept = _filter_by_distance(xy, distance)
    xy = xy[kept]

    if len(xy) < 10:
        return None

    return xy


def _filter_by_distance(xy: np.ndarray, min_dist: float) -> np.ndarray:
    if len(xy) == 0:
        return np.array([], dtype=bool)

    # Compute displacement between every consecutive pair
    deltas = np.diff(xy, axis=0)
    dists  = np.sqrt((deltas ** 2).sum(axis=1))

    # Cumulative distance travelled along the path
    cumdist = np.concatenate([[0.0], np.cumsum(dists)])

    # Keep a point whenever we've travelled >= min_dist since the last kept point
    kept    = [0]
    last_d  = 0.0
    # This loop is over *kept* points (much shorter than N), not all N rows
    thresholds = cumdist
    for i in range(1, len(xy)):
        if thresholds[i] - last_d >= min_dist:
            kept.append(i)
            last_d = thresholds[i]

    return np.array(kept, dtype=np.int64)


def plot_sample(data: np.ndarray) -> None:
    plt.figure(figsize=(8, 1))
    plt.scatter(data[:, 0], data[:, 1], s=0.2, c='blue', alpha=0.8)
    plt.axis('equal')
    plt.axis('off')
    plt.gca().invert_yaxis()
    plt.show()


def process_samples(file_paths: list[str], distance: int = 100, n_workers: int = 4, out_dir: str | None = None) -> list[np.ndarray]:
    if out_dir:
        Path(out_dir).mkdir(exist_ok=True)

    results = {}
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {
            pool.submit(_worker, p, distance, out_dir): p
            for p in file_paths
        }
        for fut in tqdm(as_completed(futures), total=len(file_paths)):
            path = futures[fut]
            try:
                results[path] = fut.result()
            except Exception as e:
                print(f"Failed: {path} — {e}")
                results[path] = None

    # Return in original order, filtering out None
    return [results[p] for p in file_paths if results.get(p) is not None]


def _worker(path: str, distance: int, out_dir: str | None):
    """Runs in a subprocess — must be a module-level function (pickle requirement)."""
    arr = process_sample(path, distance=distance)
    if arr is None:
        return None

    # Normalize
    arr = arr - arr[0]
    scale = np.max(np.abs(arr)) + 1e-8
    arr = arr / scale

    if out_dir:
        filename = f"csv_{Path(path).parent.parent.name.split('_')[-1]}_{Path(path).stem.split('_', 1)[1]}"
        np.savez(
            Path(out_dir) / f"{filename}.npz",
            points=arr,
            gt_order=np.arange(len(arr), dtype=np.int64)
        )
        return filename
    return arr

In [ ]:
sample_paths = open('/content/samples.txt').read().splitlines()
data = process_samples(sample_paths, out_dir='/content/processed/')

0it [00:00, ?it/s]


#### Data Loader

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

class HandwritingDataset(Dataset):
    def __init__(self, npz_dir: str):
        self.files = list(Path(npz_dir).glob('*.npz'))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        points   = torch.tensor(data['points'],   dtype=torch.float32)
        gt_order = torch.tensor(data['gt_order'], dtype=torch.long)

        return points, gt_order

def collate_fn(batch):
    points_list, order_list = zip(*batch)

    lengths = torch.tensor([len(p) for p in points_list])

    # Pad points with 0.0, gt_order with -1 (so loss can ignore pad positions)
    points_padded = torch.zeros(len(batch), lengths.max(), 2)
    orders_padded = torch.full((len(batch), lengths.max()), fill_value=-1, dtype=torch.long)

    for i, (pts, ord_) in enumerate(zip(points_list, order_list)):
        n = len(pts)
        points_padded[i, :n] = pts
        orders_padded[i, :n] = ord_

    return points_padded, orders_padded, lengths

In [ ]:
dataset = HandwritingDataset('/content/processed/')

# 80/20 train-val split
val_size   = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

In [ ]:
points, gt_order, lengths = next(iter(train_loader))

print("points shape :", points.shape)    # (32, N_max, 2)
print("gt_order shape:", gt_order.shape) # (32, N_max)
print("lengths      :", lengths[:8])     # first 8 sequence lengths
print("padding check:", (gt_order[0] == -1).sum().item(), "pad positions in sample 0")

#### Old

In [ ]:
sample = process_sample('/content/hwd/Dataset/Data_2000/csv/csv_0005_2.csv')
print(len(sample))
plot_sample(sample)

In [ ]:
binned = pd.cut(data['pen_down'], bins=10)

# Count per bin
counts = binned.value_counts().sort_index()
print(counts)

In [ ]:
data = pd.read_csv('/content/hwd/Dataset/Data_1500/csv/csv_0000_0.csv')
data.head()[['X cood.','Y cood.','Time']]

In [ ]:
import pandas as pd

df = pd.read_csv('/content/hwd/main_1500.csv')
print(df.shape)
df.head(2)

### Model

#### Network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PointerNetwork(nn.Module):
  def __init__(self, d_model=128, nhead=4, num_layers=3, dropout=0.1):
      super().__init__()

      self.input_proj = nn.Linear(2, d_model)

      # Transformer encoder — lets every point look at every other point
      encoder_layer = nn.TransformerEncoderLayer(
          d_model=d_model,
          nhead=nhead,
          dropout=dropout,
          batch_first=True
      )
      self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

      # Decoder memory — tracks "what have I picked so far"
      self.decoder_gru = nn.GRUCell(d_model, d_model)

      # Scoring: compares decoder state against each encoder output
      self.attn_W = nn.Linear(d_model, d_model)
      self.attn_V = nn.Linear(d_model, 1)

  def forward(self, points, lengths, target_order=None):
      B, N, _ = points.shape

      # --- Encoder ---
      pad_mask = torch.arange(N, device=points.device).unsqueeze(0) >= lengths.unsqueeze(1)

      h = self.input_proj(points)
      enc = self.encoder(h, src_key_padding_mask=pad_mask)
      enc = enc.masked_fill(pad_mask.unsqueeze(-1), 0.0)

      # --- Decoder ---
      mask_float = (~pad_mask).unsqueeze(-1).float()
      decoder_h = (enc * mask_float).sum(dim=1) / mask_float.sum(dim=1)

      logits_seq = []
      chosen = pad_mask.clone().detach()

      max_steps = lengths.max().item()
      for step in range(max_steps):
          scores = self.attn_V(
              torch.tanh(self.attn_W(enc) + decoder_h.unsqueeze(1))
          ).squeeze(-1)

          # Use -1e9 instead of -inf to prevent gradient anomalies
          scores = scores.masked_fill(chosen, -1e9)
          logits_seq.append(scores)

          # --- Teacher Forcing ---
          if target_order is not None:
              idx = target_order[:, step].clone()
              # Handle padding (-1) so scatter doesn't throw an out-of-bounds error
              idx[idx == -1] = 0
          else:
              # Inference mode: use model's own predictions
              idx = scores.argmax(dim=-1)

          chosen = chosen.scatter(1, idx.unsqueeze(1), True)

          picked = enc[torch.arange(B), idx]
          decoder_h = self.decoder_gru(picked, decoder_h)

      return torch.stack(logits_seq, dim=1)

  def decode(self, points, lengths):
    """
    Clean inference — greedy decoding with no-repeat masking.
    Returns predicted index sequence for each sample in the batch.
    """
    self.eval()
    B, N, _ = points.shape

    with torch.no_grad():
        pad_mask = torch.arange(N, device=points.device).unsqueeze(0) >= lengths.unsqueeze(1)
        h   = self.input_proj(points)
        enc = self.encoder(h, src_key_padding_mask=pad_mask)
        enc = enc.masked_fill(pad_mask.unsqueeze(-1), 0.0)

        mask_float = (~pad_mask).unsqueeze(-1).float()
        decoder_h  = (enc * mask_float).sum(dim=1) / mask_float.sum(dim=1)

        chosen    = pad_mask.clone()
        max_steps = lengths.max().item()
        output    = []

        for step in range(max_steps):
            scores = self.attn_V(
                torch.tanh(self.attn_W(enc) + decoder_h.unsqueeze(1))
            ).squeeze(-1)

            scores    = scores.masked_fill(chosen, float('-inf'))
            idx       = scores.argmax(dim=-1)          # (B,) — greedy pick
            output.append(idx)

            chosen    = chosen.scatter(1, idx.unsqueeze(1), True)
            picked    = enc[torch.arange(B), idx]
            decoder_h = self.decoder_gru(picked, decoder_h)

    return torch.stack(output, dim=1)

In [ ]:
def pointer_loss(logits, gt_order):
    B, S, N = logits.shape

    target = gt_order[:, :S]

    logits_flat = logits.reshape(B * S, N)
    target_flat = target.reshape(B * S)

    return F.cross_entropy(logits_flat, target_flat, ignore_index=-1)

#### Training

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model = PointerNetwork(d_model=128, nhead=4, num_layers=3).to(device)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Halves the LR if val loss stops improving for 3 epochs
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

def run_epoch(loader, training: bool):
    model.train() if training else model.eval()
    total_loss = 0

    with torch.set_grad_enabled(training):
        for points, gt_order, lengths in loader:
            points   = points.to(device)
            gt_order = gt_order.to(device)
            lengths  = lengths.to(device)

            # Pass gt_order to enable Teacher Forcing
            logits = model(points, lengths, target_order=gt_order)
            loss   = pointer_loss(logits, gt_order)

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 100
best_val_loss = float('inf')
patience = 5
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, training=True)
    val_loss   = run_epoch(val_loader,   training=False)

    # 1. Update the scheduler
    scheduler.step(val_loss)

    # 2. Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save the best version of the weights
        torch.save(model.state_dict(), '/content/drive/MyDrive/TUKL/handwriting.pt')
        # Reset the counter because we found a better model
        epochs_without_improvement = 0
        print(f"Epoch {epoch+1:02d} | New best val_loss: {val_loss:.4f} - Saved model")
    else:
        # Increment the counter if the loss didn't get better
        epochs_without_improvement += 1
        print(f"Epoch {epoch+1:02d} | No improvement for {epochs_without_improvement} epochs")

    print(f"Epoch {epoch+1:02d} | train: {train_loss:.4f} | val: {val_loss:.4f}")

    # 3. Trigger the exit
    if epochs_without_improvement >= patience:
        print(f"\n[Early Stopping] No improvement in validation loss for {patience} consecutive epochs. Training stopped.")
        break

### Inference

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PointerNetwork(d_model=128, nhead=4, num_layers=3).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/TUKL/handwriting.pt'))
model.to(device)

points, gt_order, lengths = next(iter(val_loader))
sample_points = points[0:1].to(device)
sample_length = lengths[0:1].to(device)

predicted_order = model.decode(sample_points, sample_length)
predicted_order = predicted_order[0].cpu().numpy()

# Reorder the points according to predicted sequence
reordered = sample_points[0].cpu().numpy()[predicted_order]

NameError: name 'val_loader' is not defined

#### Functions

In [ ]:
import numpy as np
import plotly.graph_objects as go

def visualize_ordering(model, points, gt_order, lengths, sample_idx=0, device='cuda'):
    # --- predictions ---
    sample_points = points[sample_idx:sample_idx+1].to(device)
    sample_length = lengths[sample_idx:sample_idx+1].to(device)

    predicted_order = model.decode(sample_points, sample_length)
    predicted_order = predicted_order[0].cpu().numpy()

    gt = gt_order[sample_idx].numpy()
    gt = gt[gt != -1]

    xy = points[sample_idx].numpy()[:len(gt)]

    wrong_mask = predicted_order != gt

    # --- split into correct / wrong ---
    correct_idx = ~wrong_mask
    wrong_idx   =  wrong_mask

    def hover(steps, gt_pts, pred_pts):
        return [
            f"Step: {s}<br>GT point: {g}<br>Pred point: {p}"
            for s, g, p in zip(steps, gt_pts, pred_pts)
        ]

    steps     = np.arange(len(gt))
    gt_pts    = gt
    pred_pts  = predicted_order
    x         = xy[predicted_order, 0]
    y         = -xy[predicted_order, 1]   # flip y for display

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x[correct_idx], y=y[correct_idx],
        mode='markers',
        name='Correct',
        marker=dict(color='steelblue', size=6),
        text=hover(steps[correct_idx], gt_pts[correct_idx], pred_pts[correct_idx]),
        hovertemplate='%{text}<extra></extra>',
    ))

    fig.add_trace(go.Scatter(
        x=x[wrong_idx], y=y[wrong_idx],
        mode='markers',
        name='Wrong order',
        marker=dict(color='crimson', size=8),
        text=hover(steps[wrong_idx], gt_pts[wrong_idx], pred_pts[wrong_idx]),
        hovertemplate='%{text}<extra></extra>',
    ))

    accuracy = correct_idx.sum() / len(gt) * 100
    fig.update_layout(
        title=f"Sample {sample_idx} — Accuracy: {accuracy:.1f}% ({correct_idx.sum()}/{len(gt)} correct)",
        xaxis=dict(visible=False, scaleanchor='y'),
        yaxis=dict(visible=False),
        plot_bgcolor='white',
        height=300,
        margin=dict(l=20, r=20, t=40, b=20),
    )

    fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

def animate_ordering(model, points, gt_order, lengths, sample_idx=0, device='cuda'):
    # --- predictions ---
    sample_points = points[sample_idx:sample_idx+1].to(device)
    sample_length = lengths[sample_idx:sample_idx+1].to(device)

    predicted_order = model.decode(sample_points, sample_length)
    predicted_order = predicted_order[0].cpu().numpy()

    gt = gt_order[sample_idx].numpy()
    gt = gt[gt != -1]
    n  = len(gt)

    xy = points[sample_idx].numpy()[:n]

    # GT points in their correct order
    gt_xy   = xy[gt]
    # Predicted points in the order the model chose
    pred_xy = xy[predicted_order]

    # Offset the prediction panel below GT
    y_offset = gt_xy[:, 1].min() - pred_xy[:, 1].max() - 0.15

    # --- build frames ---
    # Each frame adds one more point to both panels simultaneously
    frames = []
    for i in range(1, n + 1):
        frame_data = [
            # GT panel — points revealed in GT order
            go.Scatter(
                x=gt_xy[:i, 0],
                y=-gt_xy[:i, 1],
                mode='markers',
                marker=dict(
                    color=list(range(i)),
                    colorscale='Blues',
                    size=6,
                    showscale=False,
                ),
                name='Ground Truth',
            ),
            # Prediction panel — points revealed in predicted order
            go.Scatter(
                x=pred_xy[:i, 0],
                y=-pred_xy[:i, 1] + y_offset,
                mode='markers',
                marker=dict(
                    color=list(range(i)),
                    colorscale='Reds',
                    size=6,
                    showscale=False,
                ),
                name='Predicted',
            ),
        ]
        frames.append(go.Frame(data=frame_data, name=str(i)))

    # --- initial (empty) traces ---
    fig = go.Figure(
        data=[
            go.Scatter(x=[], y=[], mode='markers', name='Ground Truth',
                       marker=dict(color=[], colorscale='Blues', size=6)),
            go.Scatter(x=[], y=[], mode='markers', name='Predicted',
                       marker=dict(color=[], colorscale='Reds',  size=6)),
        ],
        frames=frames,
    )

    # --- panel labels as annotations ---
    fig.update_layout(
        title=f"Sample {sample_idx} — {n} points",
        height=500,
        plot_bgcolor='white',
        xaxis=dict(visible=False, scaleanchor='y'),
        yaxis=dict(visible=False),
        margin=dict(l=20, r=20, t=50, b=20),
        annotations=[
            dict(x=0.5, y=1.02, xref='paper', yref='paper',
                 text='Ground Truth', showarrow=False,
                 font=dict(size=13, color='steelblue')),
            dict(x=0.5, y=0.48, xref='paper', yref='paper',
                 text='Predicted', showarrow=False,
                 font=dict(size=13, color='crimson')),
        ],
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            y=0,
            x=0.5,
            xanchor='center',
            buttons=[
                dict(
                    label='▶ Play',
                    method='animate',
                    args=[None, dict(
                        frame=dict(duration=40, redraw=True),
                        fromcurrent=True,
                        mode='immediate',
                    )],
                ),
                dict(
                    label='⏸ Pause',
                    method='animate',
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        mode='immediate',
                    )],
                ),
            ],
        )],
        sliders=[dict(
            steps=[
                dict(method='animate', args=[[str(i)],
                     dict(mode='immediate', frame=dict(duration=40, redraw=True))],
                     label=str(i))
                for i in range(1, n + 1)
            ],
            x=0.1, len=0.8, y=-0.02,
            currentvalue=dict(prefix='Points drawn: ', visible=True),
        )],
    )

    fig.show()

In [ ]:
points, gt_order, lengths = next(iter(val_loader))
animate_ordering(model, points, gt_order, lengths, sample_idx=20, device=device)